# Challenge 03 - Solar Park

## Data Preparation and Loading

The goal is to predict the power produced by the photovoltaic plant (960 kWp, Lecce) during the **next hour**, minimizing the RMSE on the test set. The dataset contains hourly averages with the columns `Time Frame` (hour ordinal, 1-24), `Date` (string, format `%m/%d/%y %I:%M %p`), `Ta (C)` and `Tm (C)` (ambient and module temperature), `I3 (W/m2)` and `I15 (W/m2)` (irradiance on panels tilted by 3 and 15 degrees) and the target `P (kW)`, which is not available in the test set.

The patterns are already in temporal order and are never shuffled or sorted. Each training pattern is built as `(features at time t, power at time t+1)`: only features at time `t` are used, and the power at time `t` is never included in the input vector. The dataset is not augmented; the only reduction is the removal of pairs whose two `Time Frame` values are not consecutive (the target would not be the *next hour*).

The hour of the day is taken from `Time Frame`, not from the time part of the `Date` string, because that part is unreliable in some rows (for instance different `Time Frame` values sharing the same timestamp). `Date` is only used for the day of the year and the month. Hour, day of year and month are encoded cyclically. Any preprocessing is fitted on the training folds only (it lives inside a `Pipeline`), and the test set is only transformed.

In [9]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, VotingRegressor

# Configuration
train_path = '../DBs/SolarPark/train.txt'
test_path = '../DBs/SolarPark/test.txt'
result_path_solution1 = '../pred_sol1.txt'
result_path_solution2 = '../pred_sol2.txt'
result_path_solution3 = '../pred_sol3.txt'
result_path_final = '../Es4Predictions.txt'

# Column names (header of the CSV files)
time_frame_col = 'Time Frame'   # hour ordinal, 1..24
date_col = 'Date'               # string, e.g. '3/5/12 4:55 PM'
date_format = '%m/%d/%y %I:%M %p'
feature_cols = ['Ta (C)', 'Tm (C)', 'I3 (W/m2)', 'I15 (W/m2)']
target_col = 'P (kW)'           # not available in the test set

train_dataframe = pd.read_csv(train_path)
test_dataframe = pd.read_csv(test_path)

assert {time_frame_col, date_col, *feature_cols, target_col} <= set(train_dataframe.columns)
assert {time_frame_col, date_col, *feature_cols} <= set(test_dataframe.columns)

print('Train shape:', train_dataframe.shape)
print('Test shape:', test_dataframe.shape)
print(train_dataframe.head())

Train shape: (7902, 7)
Test shape: (7900, 6)
   Time Frame            Date  Ta (C)  Tm (C)  I3 (W/m2)  I15 (W/m2)  P (kW)
0          16  3/5/12 4:55 PM   13.01   12.05      52.47       53.08    6.50
1          17  3/5/12 5:55 PM   11.81    8.50      10.41       10.95    0.01
2          18  3/5/12 6:55 PM   10.33    5.68       0.00        0.00    0.00
3          19  3/5/12 7:55 PM    9.66    4.67       0.00        0.00    0.00
4          20  3/5/12 8:55 PM    8.91    3.82       0.00        0.00    0.00


In [10]:
def save_predictions(predictions, path):
    np.savetxt(path, predictions)
    print(f'Predictions successfully saved to {path}')

In [11]:
def cyclical(values, period):
    """Circular encoding of a periodic quantity as (sin, cos)."""
    angle = 2 * np.pi * np.asarray(values, dtype=np.float32) / period
    return np.sin(angle), np.cos(angle)

def build_features(dataframe):
    """Features at time t only: raw measurements + cyclical hour of day, day of year and month.

    The hour comes from 'Time Frame' (the time part of 'Date' contains errors);
    'Date' is only used for the day of the year and the month.
    """
    dates = pd.to_datetime(dataframe[date_col], format=date_format)
    time_frame = dataframe[time_frame_col].to_numpy()
    raw = dataframe[feature_cols].to_numpy(dtype=np.float32)
    hour_sin, hour_cos = cyclical(time_frame, 24)
    doy_sin, doy_cos = cyclical(dates.dt.dayofyear, 366)
    month_sin, month_cos = cyclical(dates.dt.month, 12)
    x = np.column_stack([raw, hour_sin, hour_cos, doy_sin, doy_cos, month_sin, month_cos])
    return x.astype(np.float32), time_frame

def make_predict_ahead(dataframe):
    """Pairs (x at time t, y at time t+1), in the original temporal order.

    The target is shifted by one row; a pair is kept only if the 'Time Frame'
    of the next row follows the current one (1..24, then back to 1).
    """
    x, time_frame = build_features(dataframe)
    y = dataframe[target_col].to_numpy(dtype=np.float32)
    consecutive = time_frame[1:] == (time_frame[:-1] % 24) + 1
    valid = consecutive & np.isfinite(y[1:]) & np.isfinite(x[:-1]).all(axis=1)
    print('Valid (t, t+1) pairs: %d / %d' % (valid.sum(), len(valid)))
    return x[:-1][valid], y[1:][valid]

In [12]:
# Training patterns for the predict ahead scenario
train_x, train_y = make_predict_ahead(train_dataframe)
print('train_x:', train_x.shape, 'train_y:', train_y.shape)

# Test patterns: same features, original row order (one prediction per row)
test_x, _ = build_features(test_dataframe)
print('test_x:', test_x.shape)
assert test_x.shape[1] == train_x.shape[1]

Valid (t, t+1) pairs: 7883 / 7901
train_x: (7883, 10) train_y: (7883,)
test_x: (7900, 10)


## Solution 1

In [13]:
# TODO

### Solution 1 Prediction and Export

In [14]:
# TODO